In [ ]:
from ultralytics import YOLOE
from ultralytics.models.yolo.yoloe import YOLOEPETrainer

In [ ]:
model = YOLOE("yoloe-11m.yaml")      # architettura DETECTION dal config (niente -seg)
model.load("yoloe-11m-seg.pt")   # usa un checkpoint -det se disponibile (VistaCrash è bbox-only, niente maschere)
model.train(
    data="./data/VistaCrash/data.yaml",
    trainer=YOLOEPETrainer,          # linear probing
    epochs=50,
    lr0=1e-2,
    optimizer="AdamW",
    weight_decay=0.025,
    momentum=0.9,
    batch=64,                        # scala alla tua VRAM (su 16-24 GB scendi a 16-32)
    close_mosaic=5,
    imgsz=1280,                      # VistaCrash è annotato a 1280×960; usa risoluzione alta per gli oggetti aerei piccoli
    name="vistacrash_lp",
)

In [ ]:
from ultralytics import YOLO
m = YOLO("runs/detect/vistacrash_lp/weights/best.pt")
print(m.names)   # deve essere {0: 'crashed car', 1: 'person', 2: 'car'}

In [ ]:
from ultralytics import YOLO
m = YOLO("./runs/detect/")  # prendi best.pt dalla cartella della run
metrics = m.val(data="data/VistaCrash/data.yaml", split="test", imgsz=1280)
print("mAP50   :", metrics.box.map50)
print("mAP50-95:", metrics.box.map)
print("per-classe mAP50-95:", metrics.box.maps)   # array indicizzato per classe
print(m.names)
# la confusion matrix viene salvata come confusion_matrix.png nella cartella della run